# Account Data Analysis

This notebook walks through the account generator, generates a full account population linked to customers, and analyzes the output to verify the distribution and relationships look realistic.

**What we cover:**
1. Generate customers and accounts
2. Inspect a sample record for each account type
3. Account type distribution
4. Accounts per customer
5. Balance distribution by account type
6. Interest rate distribution by account type
7. CD term breakdown
8. Account open date spread
9. Export account data

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns
from datetime import date

from src.generators.customer import generate_customers
from src.generators.accounts import generate_accounts
from src.models.account import AccountType

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print('Setup complete.')

## 1. Generate Customers and Accounts

Accounts are always generated from a customer population — every account has a `customer_id` foreign key. We use the same seed as the customer notebook so the two datasets are consistent with each other.

In [ ]:
SEED = 42

customers = generate_customers(count=250, seed=SEED)
accounts  = generate_accounts(customers=customers, total_accounts=450, seed=SEED)

print(f'Customers : {len(customers):,}')
print(f'Accounts  : {len(accounts):,}')

## 2. Inspect Sample Records

One record per account type so we can verify the fields look correct — particularly the CD-specific fields (`cd_term_months`, `cd_maturity_date`) which should be populated only on CD accounts.

In [ ]:
def print_account(acc):
    print(f'  Account ID     : {acc.account_id}')
    print(f'  Customer ID    : {acc.customer_id}')
    print(f'  Type           : {acc.account_type}')
    print(f'  Account Number : {acc.account_number}')
    print(f'  Routing Number : {acc.routing_number}')
    print(f'  Status         : {acc.status}')
    print(f'  Open Date      : {acc.open_date}')
    print(f'  Balance        : ${acc.balance:,.2f}')
    print(f'  Interest Rate  : {acc.interest_rate:.2%}')
    print(f'  CD Term        : {acc.cd_term_months} months' if acc.cd_term_months else '  CD Term        : N/A')
    print(f'  CD Maturity    : {acc.cd_maturity_date}' if acc.cd_maturity_date else '  CD Maturity    : N/A')

for acct_type in AccountType:
    sample = next(a for a in accounts if a.account_type == acct_type)
    print(f'--- {acct_type} ---')
    print_account(sample)
    print()

## 3. Build a DataFrame

In [ ]:
today = date.today()

records = []
for a in accounts:
    records.append({
        'account_id':      a.account_id,
        'customer_id':     a.customer_id,
        'account_type':    a.account_type,
        'account_number':  a.account_number,
        'status':          a.status,
        'open_date':       a.open_date,
        'account_age_days': (today - a.open_date).days,
        'balance':         a.balance,
        'interest_rate':   a.interest_rate,
        'interest_rate_pct': round(a.interest_rate * 100, 4),
        'cd_term_months':  a.cd_term_months,
        'cd_maturity_date': a.cd_maturity_date,
    })

df = pd.DataFrame(records)
print(f'DataFrame shape: {df.shape}')
df.head()

## 4. Account Type Distribution

Expected mix: ~45% Checking, ~35% Savings, ~20% CD. Every customer is guaranteed at least one Checking account, so Checking will always be the dominant type.

In [ ]:
type_counts = df['account_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted', 3), edgecolor='white')
axes[0].set_title('Account Count by Type', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(type_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=11)

axes[1].pie(
    type_counts.values,
    labels=type_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('muted', 3),
)
axes[1].set_title('Account Type Share', fontsize=13)

plt.tight_layout()
plt.show()

## 5. Accounts Per Customer

How many accounts does each customer hold? Every customer has at least 1 (their guaranteed Checking account). The distribution shows how the remaining accounts were spread across the population.

In [ ]:
accounts_per_customer = df.groupby('customer_id')['account_id'].count()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution histogram
axes[0].hist(accounts_per_customer.values, bins=range(1, accounts_per_customer.max() + 2),
             color='steelblue', edgecolor='white', align='left')
axes[0].set_title('Accounts Per Customer', fontsize=13)
axes[0].set_xlabel('Number of Accounts')
axes[0].set_ylabel('Number of Customers')
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(1))

# Summary stats
axes[1].axis('off')
stats = accounts_per_customer.describe()
stats_text = '\n'.join([
    f"Total customers : {len(accounts_per_customer):,}",
    f"Min accounts    : {int(stats['min'])}",
    f"Max accounts    : {int(stats['max'])}",
    f"Mean accounts   : {stats['mean']:.2f}",
    f"Median accounts : {stats['50%']:.1f}",
])
axes[1].text(0.2, 0.5, stats_text, transform=axes[1].transAxes,
             fontsize=13, verticalalignment='center', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.8))
axes[1].set_title('Summary', fontsize=13)

plt.tight_layout()
plt.show()

## 6. Balance Distribution by Account Type

Expected ranges:
- **Checking**: \$250 – \$15,000
- **Savings**: \$500 – \$50,000
- **CD**: \$1,000 – \$100,000

CDs should have the highest balances on average — customers lock money into CDs precisely because they have larger sums to invest.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = ['Checking', 'Savings', 'CD']
sns.boxplot(data=df, x='account_type', y='balance', order=order,
            palette='muted', ax=axes[0])
axes[0].set_title('Balance Range by Account Type', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('Balance (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

for acct_type, color in zip(order, sns.color_palette('muted', 3)):
    subset = df[df['account_type'] == acct_type]['balance']
    axes[1].hist(subset, bins=20, alpha=0.6, label=acct_type, color=color, edgecolor='white')
axes[1].set_title('Balance Distribution Overlay', fontsize=13)
axes[1].set_xlabel('Balance (USD)')
axes[1].set_ylabel('Count')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
axes[1].legend()

plt.tight_layout()
plt.show()

df.groupby('account_type')['balance'].agg(['min', 'median', 'max', 'mean']) \
  .map(lambda x: f'${x:,.2f}') \
  .reindex(order)

## 7. Interest Rate Distribution by Account Type

Expected APY ranges:
- **Checking**: 0.01% – 0.5% (most checking accounts pay very little)
- **Savings**: 1.0% – 5.5% (HYSA range)
- **CD**: 3.0% – 5.8% (higher locked-in rate reward)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for acct_type, color in zip(order, sns.color_palette('muted', 3)):
    subset = df[df['account_type'] == acct_type]['interest_rate_pct']
    ax.hist(subset, bins=20, alpha=0.65, label=acct_type, color=color, edgecolor='white')

ax.set_title('Interest Rate Distribution by Account Type (APY %)', fontsize=13)
ax.set_xlabel('Annual Percentage Yield (%)')
ax.set_ylabel('Count')
ax.legend()

plt.tight_layout()
plt.show()

df.groupby('account_type')['interest_rate_pct'].agg(['min', 'median', 'max', 'mean']) \
  .map(lambda x: f'{x:.4f}%') \
  .reindex(order)

## 8. CD Term Breakdown

CD accounts should be distributed across 5 standard term lengths: 3, 6, 12, 24, and 36 months.

In [ ]:
cd_df = df[df['account_type'] == 'CD'].copy()
term_counts = cd_df['cd_term_months'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

term_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f'CD Term Distribution (n={len(cd_df)})', fontsize=13)
axes[0].set_xlabel('Term (Months)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(term_counts.values):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontsize=11)

# Balance vs term — do longer-term CDs tend to have higher balances?
sns.boxplot(data=cd_df, x='cd_term_months', y='balance',
            order=sorted(cd_df['cd_term_months'].unique()), palette='muted', ax=axes[1])
axes[1].set_title('CD Balance by Term Length', fontsize=13)
axes[1].set_xlabel('Term (Months)')
axes[1].set_ylabel('Balance (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

## 9. Account Open Date Spread

Account open dates are sampled between the customer's `customer_since` date and today, so accounts can't predate the customer relationship. The spread should cover the full history window with no obvious gaps.

In [ ]:
df['open_year'] = pd.to_datetime(df['open_date']).dt.year

fig, ax = plt.subplots(figsize=(14, 5))

for acct_type, color in zip(order, sns.color_palette('muted', 3)):
    subset = df[df['account_type'] == acct_type]
    year_counts = subset['open_year'].value_counts().sort_index()
    ax.plot(year_counts.index, year_counts.values, marker='o', label=acct_type, color=color)

ax.set_title('Account Openings Per Year by Type', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Accounts Opened')
ax.legend()
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.tight_layout()
plt.show()

print(f'Earliest open date : {df["open_date"].min()}')
print(f'Latest open date   : {df["open_date"].max()}')

## 10. Export Account Data

In [ ]:
from src.utils.exporters import write_json, write_csv

output_dir = Path.cwd().parent / 'data' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

write_json(accounts, output_dir / 'accounts.json')
write_csv(accounts, output_dir / 'accounts.csv')

print(f'Files written to: {output_dir.resolve()}')